# HALO lens flare (Reflective) → Kaggle dataset

Notebook này **không train**. Nó lấy các mẫu **Reflective** (bóng ma: ánh sáng phản xạ qua lại giữa các
mặt thấu kính) của dataset [HALO](https://huggingface.co/datasets/jdzhang0929/halo-flare-dataset) trên
Hugging Face, thu nhỏ từ 4K về 1280×720 rồi tạo một Kaggle dataset **riêng tư**.

| | HALO gốc | dataset này (mặc định) |
|---|---|---|
| mẫu | 4.945: Streak 1.656, Reflective 1.655, Glare 817, Shimmer 817 | **1.655 Reflective**, đủ 32 scene |
| ảnh | 3840×2160 RGBA PNG | **1280×720 RGB PNG** (Lanczos, thu đúng 3 lần) |
| dung lượng | 152,8 GB, 71 shard WebDataset | ~4,3 GB, mỗi scene một tar |
| phải tải | 152,8 GB | **~51 GB**: chỉ các member Reflective |

Mỗi mẫu là bộ ba cùng một khung hình: `gt.png` (cảnh không lóe), `flare.png` (có lóe),
`separate.png` (chỉ lớp lóe), kèm một `.json` metadata. Cell 3 in con số thật trước khi tải ảnh.

**Khác gì đoạn script tải theo shard.** Có ba chỗ đo được:
1. **`sample_id` không duy nhất.** Trong `finalist.json`, 1.655 mẫu Reflective chỉ có 889 `sample_id`:
   cùng scene, hiệu ứng và camera nhưng khác HDRI nền hoặc cấu hình đèn. Lưu theo `{sample_id}.{phần}`
   thì gần một nửa số mẫu đè lên nhau mà không báo lỗi. Ở đây `uid = sample_id + "_" + final_idx`,
   và `final_idx` (0–4944) là duy nhất.
2. **New Dataset từ Output chỉ nhận 500 file đầu** (đo khi dựng tartanair640). 1.655 mẫu × 4 file là
   6.620 file, nên Cell 6 đóng gói mỗi scene thành một tar. Kaggle giải nén mỗi tar thành thư mục cùng tên.
3. **Không tải nguyên shard.** Mỗi shard ~2,2 GB chứa ~70 mẫu đã trộn, Reflective chỉ chiếm 1/3.
   `tarfile` đọc shard qua HTTP Range: nó tải header 512 byte của từng member và chỉ tải phần dữ liệu
   của member Reflective.

Đo trên một mẫu (`Scene049_Reflective014_camera11`): alpha bằng 255 ở cả ba ảnh nên bị bỏ (Cell 5
kiểm lại trên mọi ảnh). `flare` **không** đúng bằng `gt` cộng `separate`: sai số trung bình là 1,6/255
với screen blend, 1,8 khi cộng tuyến tính và 2,4 khi cộng trên sRGB, so với 3,0 nếu coi `flare = gt`.
Vì vậy `separate` chỉ là lớp lóe gần đúng, không phải phần dư chính xác.

**Giấy phép.** HALO dùng CC BY-NC-SA 4.0, chỉ cho nghiên cứu phi thương mại. Điều khoản để được tải:
trích dẫn UniSER khi công bố, không phân phối lại dưới điều khoản thoáng hơn, ghi nguồn HDRI. Dataset
tạo ra để riêng tư với license CC-BY-NC-SA-4.0, kèm `README.md` ghi nguồn, trích dẫn và các thay đổi.

## Chạy
1. *Settings*: **Accelerator: None** (CPU, không tốn quota GPU), **Internet: On**.
2. *Add-ons → Secrets*: bật **`HF_TOKEN`** cho notebook này. Tài khoản HF của token phải đã điền form và
   bấm *I agree, request access* trên trang HALO (duyệt tự động); chưa có quyền thì Cell 3 dừng và báo.
3. Tuỳ chọn: thêm secret **`KAGGLE_API_TOKEN`** (nội dung `~/.kaggle/access_token`, hoặc Kaggle →
   Settings → API → *Generate New Token*). Có secret này thì Cell 7 tự tạo dataset. Không có thì tạo tay
   từ tab *Output*.
4. **Chạy thử** (không bắt buộc): `PILOT = True` → *Run All*. Lượt này lấy 3 mẫu từ 1 shard để xem tốc độ
   tải và giải mã trên Kaggle. Lượt này không tạo dataset.
5. **Chạy thật**: `PILOT = False` → *Save Version → Save & Run All (Commit)*. Dùng Commit, đừng dùng phiên
   tương tác: phiên tương tác có thể bị Kaggle khởi động lại, và khi đó `/kaggle/working` trống.
   Thời gian chưa đo trên Kaggle. Ước tính là 20–60 phút: tải ~51 GB cộng giải mã 5.000 PNG 4K trên 4 lõi.

## Cell 1 — Lựa chọn

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

EFFECTS = ('Reflective',)  # bóng ma; thêm 'Streak', 'Glare', 'Shimmer' nếu muốn (cả 4 ≈ 13 GB output)
MAX_SIDE = 1280            # 3840×2160 → 1280×720
N_SHARDS = None            # None = cả 71 shard. Shard đã trộn scene, nên 10 shard ≈ 230 mẫu từ đủ các scene
PILOT = False              # True: 3 mẫu từ 1 shard, không tạo dataset, chỉ để đo tốc độ
WORKERS = 2 * (os.cpu_count() or 4)  # process: một nửa thời gian chờ mạng, một nửa giải mã PNG 4K
KAGGLE_OWNER = 'buidinhkhoi'
NAME = 'halo-' + '-'.join(effect.lower() for effect in EFFECTS) + f'-{MAX_SIDE}'
DATASET = f'{KAGGLE_OWNER}/{NAME}'   # chưa có thì Cell 7 tạo mới, có rồi thì thêm version
TITLE = f'HALO {" ".join(EFFECTS)} flare {MAX_SIDE}px'
OUT = Path('/kaggle/working') / NAME.replace('-', '_')  # nội dung dataset = thư mục này
BUILD = Path('/tmp/halo_build')  # ảnh lẻ dựng ở đây, ngoài output; Cell 6 đóng gói sang OUT
if PILOT:
    OUT, BUILD = OUT.with_name(OUT.name + '_pilot'), Path('/tmp/halo_build_pilot')

## Cell 2 — Hàm dùng chung (chỉ định nghĩa, chưa chạy gì)

In [ ]:
import csv
import hashlib
import io
import json
import multiprocessing as mp
import os
import posixpath
import shutil
import tarfile
import time
from collections import Counter
from functools import partial
from pathlib import Path

import requests
from PIL import Image

REPO = 'jdzhang0929/halo-flare-dataset'
REVISION = 'd588bc94b10960f4597ddcc155eadd354cfa304e'  # commit HF ngày 2026-06-13: cùng code ra cùng dữ liệu
PARTS = ('gt.png', 'flare.png', 'separate.png', 'json')  # thứ tự member của một mẫu trong tar
IMAGES = PARTS[:3]


def hf_url(path):
    return f'https://huggingface.co/datasets/{REPO}/resolve/{REVISION}/{path}'


def hf_json(path, token):
    """Tải một file JSON nhỏ của repo; báo rõ khi token chưa được duyệt quyền."""
    response = requests.get(hf_url(path), headers={'Authorization': f'Bearer {token}'}, timeout=120)
    if response.status_code in (401, 403):
        raise PermissionError(
            f'HF trả {response.status_code} cho {path}: tài khoản của HF_TOKEN chưa được vào {REPO}. Mở '
            f'https://huggingface.co/datasets/{REPO}, đăng nhập đúng tài khoản đó, điền form và bấm '
            '"I agree, request access" (duyệt tự động), rồi chạy lại.')
    response.raise_for_status()
    return response.json()


def effect_of(sample_id):
    """'Scene003_Reflective019_camera01' -> 'Reflective' (sample_id = scene_effectid_camera)."""
    parts = sample_id.split('_')
    return parts[1].rstrip('0123456789') if len(parts) >= 3 else None


class RemoteFile:
    """File chỉ đọc trên HTTP Range, đủ cho tarfile (read, seek, tell).

    tarfile đọc header 512 byte của từng member rồi seek qua phần dữ liệu, nên chỉ những
    member được đọc thật mới bị tải. Lần đọc nhỏ được đọc trước thành 4 KiB: trước mỗi header
    tarfile đọc 1 byte ngay trước nó (kiểm hết file), nên byte đó và header đi chung một request.
    """

    READAHEAD = 4096

    def __init__(self, path, token, session=None):
        self.path, self.token = path, token
        self.session = session or requests.Session()
        self.url = None  # URL CDN đã ký sau redirect; dùng lại tới khi hết hạn
        self.size = None
        self.pos, self.buf_start, self.buf = 0, 0, b''
        self.requests = self.bytes = 0
        self._fetch(0, 1)  # lấy self.size từ Content-Range

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = offset + (self.pos if whence == 1 else self.size if whence == 2 else 0)
        return self.pos

    def read(self, n=-1):
        n = self.size - self.pos if n is None or n < 0 else min(n, self.size - self.pos)
        if n <= 0:
            return b''
        start = self.pos - self.buf_start
        if 0 <= start and start + n <= len(self.buf):
            data = self.buf[start:start + n]
        elif n < self.READAHEAD:
            self.buf_start = self.pos
            self.buf = self._fetch(self.pos, min(self.READAHEAD, self.size - self.pos))
            data = self.buf[:n]
        else:
            data = self._fetch(self.pos, n)
        self.pos += len(data)
        return data

    def _fetch(self, start, length):
        last = None
        for attempt in range(8):
            headers = {'Range': f'bytes={start}-{start + length - 1}'}
            if self.url is None:
                headers['Authorization'] = f'Bearer {self.token}'  # requests bỏ header này khi redirect sang CDN
            try:
                with self.session.get(self.url or hf_url(self.path), headers=headers, timeout=(30, 300),
                                      stream=True) as response:
                    if response.status_code == 206:
                        data = response.content
                        if len(data) != length:
                            raise IOError(f'nhận {len(data)} byte thay vì {length}')
                        if self.url is None:
                            self.url = response.url
                        if self.size is None:
                            self.size = int(response.headers['Content-Range'].rsplit('/', 1)[1])
                        self.requests += 1
                        self.bytes += length
                        return data
                    if response.status_code in (401, 403) and self.url is None:
                        raise PermissionError(f'{self.path}: HF trả {response.status_code}')
                    if response.status_code == 200:
                        raise IOError('server bỏ qua Range')  # không tải nguyên shard 2 GB
                    if response.status_code in (400, 401, 403, 410):
                        self.url = None  # URL đã ký hết hạn: xin lại từ huggingface.co
                    last = f'HTTP {response.status_code}'
            except (requests.RequestException, IOError) as error:
                if isinstance(error, PermissionError):
                    raise
                last = error
            time.sleep(min(60, 2 ** attempt))
        raise IOError(f'{self.path} byte {start}+{length}: {last}')


def shrink(data, max_side):
    """PNG 4K -> PNG có cạnh dài max_side (Lanczos). Alpha toàn 255 thì bỏ.

    Alpha thật (nếu có) được thu nhỏ riêng: resize RGBA của PIL nhân màu với alpha trước,
    làm mất màu ở chỗ alpha nhỏ.
    """
    with Image.open(io.BytesIO(data)) as image:
        image.load()
        source = image.size
        if image.mode not in ('RGB', 'RGBA'):
            image = image.convert('RGBA' if 'A' in image.getbands() else 'RGB')
        alpha = None
        if image.mode == 'RGBA':
            channel = image.getchannel('A')
            alpha = None if channel.getextrema() == (255, 255) else channel
            image = image.convert('RGB')
        scale = max_side / max(image.size)
        if scale < 1:
            size = (round(image.width * scale), round(image.height * scale))
            image = image.resize(size, Image.LANCZOS)
            alpha = alpha.resize(size, Image.LANCZOS) if alpha is not None else None
        if alpha is not None:
            image.putalpha(alpha)
        out = io.BytesIO()
        image.save(out, 'PNG')
        return out.getvalue(), list(source), list(image.size), alpha is not None


def write_atomic(path, data):
    partial_path = path.with_name(path.name + '.part')
    partial_path.write_bytes(data)
    os.replace(partial_path, path)


def build_shard(shard, build, token, effects, max_side, limit=None):
    """Mọi mẫu thuộc `effects` của một shard -> build/<scene>/<uid>.<phần>; trả về báo cáo.

    Shard đã xong (có build/_shards/<shard>.json) thì không làm lại. uid = sample_id + final_idx:
    sample_id KHÔNG duy nhất (1655 mẫu Reflective chỉ có 889 sample_id), đặt tên theo sample_id
    thì gần một nửa số mẫu đè lên nhau.
    """
    marker = build / '_shards' / f'{shard}.json'
    if marker.exists():
        return json.loads(marker.read_text())
    started = time.perf_counter()
    remote = RemoteFile(f'shards/{shard}', token)
    seen, records, problems = Counter(), [], []
    group, key = {}, None

    def flush():
        effect = effect_of(key)
        seen[effect] += 1
        if effect not in effects:
            return
        missing = [part for part in PARTS if group.get(part) is None]
        if missing:
            problems.append(f'{shard}: {key} thiếu {missing}')
            return
        meta = json.loads(group['json'])
        if meta.get('effect_type') != effect or meta.get('sample_id') != key:
            problems.append(f'{shard}: {key} có json {meta.get("sample_id")} / {meta.get("effect_type")}')
            return
        uid = f'{key}_{int(meta["final_idx"]):04d}'
        folder = build / meta['scene']
        folder.mkdir(parents=True, exist_ok=True)
        images = {}
        for part in IMAGES:
            png, source, size, alpha = shrink(group[part], max_side)
            write_atomic(folder / f'{uid}.{part}', png)
            images[part] = {'source_size': source, 'size': size, 'alpha': alpha, 'bytes': len(png)}
        meta['source_names'] = {name: meta.pop(name) for name in ('light', 'flare', 'separate') if name in meta}
        meta.update({name: f'{uid}.{part}' for name, part in zip(('light', 'flare', 'separate'), IMAGES)})
        meta.update(uid=uid, shard=shard, images=images, resample='lanczos', max_side=max_side)
        write_atomic(folder / f'{uid}.json', json.dumps(meta, indent=1).encode())
        records.append({name: meta[name] for name in
                        ('uid', 'scene', 'effect_type', 'effect_id', 'sample_id', 'final_idx', 'orig_idx', 'shard')})

    with tarfile.open(fileobj=remote, mode='r:') as tar:
        for member in tar:
            if not member.isfile():
                continue
            stem, _, part = posixpath.basename(member.name).partition('.')
            if group and (stem != key or part in group):
                flush()
                group = {}
                if limit and len(records) >= limit:
                    break
            key = stem
            wanted = effect_of(stem) in effects and part in PARTS
            if wanted:
                remote.seek(member.offset_data)
            group[part] = remote.read(member.size) if wanted else None
        else:
            if group:
                flush()
    report = {'shard': shard, 'samples': sum(seen.values()), 'seen': dict(seen), 'records': records,
              'problems': problems, 'requests': remote.requests, 'downloaded': remote.bytes,
              'seconds': round(time.perf_counter() - started, 1), 'limit': limit}
    marker.parent.mkdir(parents=True, exist_ok=True)
    write_atomic(marker, json.dumps(report).encode())
    return report


def _build_shard_safe(shard, **kwargs):
    try:
        return build_shard(shard, **kwargs)
    except Exception as error:  # một shard lỗi không làm dừng các shard khác; lượt sau làm lại
        return {'shard': shard, 'error': f'{type(error).__name__}: {error}'}


def build_all(shards, build, token, effects, max_side, workers, limit=None, log=print):
    """Chạy build_shard song song (process: giải mã PNG tốn CPU); shard lỗi được thử lại một lượt."""
    job = partial(_build_shard_safe, build=build, token=token, effects=effects, max_side=max_side, limit=limit)
    reports, todo, started = {}, list(shards), time.perf_counter()
    for attempt in range(2):
        failed = []
        with mp.get_context('fork').Pool(min(workers, len(todo))) as pool:
            for report in pool.imap_unordered(job, todo):
                if 'error' in report:
                    failed.append(report['shard'])
                    log(f'LỖI {report["shard"]}: {report["error"]}')
                    continue
                reports[report['shard']] = report
                kept = sum(len(item['records']) for item in reports.values())
                log(f'[{len(reports)}/{len(shards)}] {report["shard"]}: {len(report["records"])} mẫu giữ '
                    f'/ {report["samples"]}, {report["downloaded"] / 1e9:.2f} GB, {report["seconds"]:.0f} s '
                    f'| tổng {kept} mẫu, {(time.perf_counter() - started) / 60:.1f} phút')
        if not failed:
            break
        todo = failed
        log(f'thử lại {len(failed)} shard lỗi')
    return reports, failed


def md5(data):
    return hashlib.md5(data).hexdigest()


def pack_scenes(build, out, remove_source, log=print):
    """Mỗi scene một tar không nén out/<scene>.tar, member đặt tên theo file (không có thư mục).

    New Dataset từ Output chỉ nhận 500 file đầu; Kaggle giải nén mỗi tar thành thư mục cùng tên,
    nên dataset đọc thành <scene>/<uid>.<phần>. Mỗi file được băm MD5 lúc ghi, tar được đọc lại
    và so MD5 trước khi đổi .tar.part -> .tar; scene chỉ bị xoá (remove_source) sau bước đó.
    """
    out.mkdir(parents=True, exist_ok=True)
    folders = sorted(path for path in build.iterdir() if path.is_dir() and not path.name.startswith('_'))
    sizes = {path.name: sum(item.stat().st_size + 1024 for item in path.iterdir()) for path in folders}
    same_disk = os.stat(build).st_dev == os.stat(out).st_dev
    needed = max(sizes.values(), default=0) if remove_source and same_disk else sum(sizes.values())
    free = shutil.disk_usage(out).free
    if free < needed + 2**30:
        raise OSError(f'{out}: còn {free / 1e9:.2f} GB, đóng gói cần {needed / 1e9:.2f} GB; chưa ghi gì')
    inventory = {}
    for folder in folders:
        target, partial_path = out / f'{folder.name}.tar', out / f'{folder.name}.tar.part'
        written = {}
        with tarfile.open(partial_path, 'w', format=tarfile.PAX_FORMAT) as tar:
            for path in sorted(folder.iterdir()):
                data = path.read_bytes()
                info = tarfile.TarInfo(path.name)
                info.size, info.mode, info.mtime = len(data), 0o644, int(path.stat().st_mtime)
                tar.addfile(info, io.BytesIO(data))
                written[path.name] = (len(data), md5(data))
        with tarfile.open(partial_path) as tar:
            read_back = {member.name: (member.size, md5(tar.extractfile(member).read())) for member in tar}
        if read_back != written:
            raise IOError(f'{partial_path}: tar đọc lại khác lúc ghi; nguồn chưa bị đụng tới')
        os.replace(partial_path, target)
        inventory.update({f'{folder.name}/{name}': value for name, value in written.items()})
        if remove_source:
            shutil.rmtree(folder)
        log(f'{folder.name}: {len(written)} file, {target.stat().st_size / 1e9:.3f} GB, đọc lại khớp MD5')
    return inventory


def render_dir(record):
    """Thư mục render gốc trong finalist.json, vd 'Scene003_02_1_4_Reflective019_aerial-suburban-city-sunny_4K'
    (chứa tên HDRI nền; hai mẫu cùng sample_id khác nhau ở thư mục này)."""
    return record['flare_image_key'].split('/')[-2]


def write_index(path, records, finalist):
    by_index = {record['_final_idx']: record for record in finalist}
    columns = ['uid', 'scene', 'effect_type', 'effect_id', 'sample_id', 'final_idx', 'orig_idx', 'shard', 'render_dir']
    with open(path, 'w', newline='') as handle:
        writer = csv.DictWriter(handle, columns)
        writer.writeheader()
        for record in sorted(records, key=lambda item: item['final_idx']):
            writer.writerow({**record, 'render_dir': render_dir(by_index[record['final_idx']])})


## Cell 3 — Token, quyền truy cập, kế hoạch (chưa tải ảnh)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except ImportError:  # ngoài Kaggle: đọc biến môi trường
    _secrets = None


secret_errors = {}


def secret(name):
    try:
        return _secrets.get_secret(name) if _secrets else os.environ.get(name)
    except Exception as error:  # chưa tick Attached cho notebook này, hoặc sai tên secret
        secret_errors[name] = f'{type(error).__name__}: {error}'
        return None


HF_TOKEN = secret('HF_TOKEN')
if not HF_TOKEN:
    raise SystemExit(f'Chưa đọc được HF_TOKEN ({secret_errors.get("HF_TOKEN", "secret rỗng")}). Add-ons → Secrets: '
                     'secret phải tên đúng HF_TOKEN và được tick Attached cho notebook này. Rồi chạy lại cell này.')
try:
    finalist = hf_json('finalist.json', HF_TOKEN)
    shard_index = hf_json('shards/shard_index.json', HF_TOKEN)
except requests.ConnectionError as error:
    raise SystemExit(f'Không kết nối được Hugging Face ({error}). Bật Settings → Internet: On rồi chạy lại.')

all_shards = [item['shard'] for item in shard_index['shards']]
shards = all_shards[:1] if PILOT else all_shards[:N_SHARDS]
whole = len(shards) == len(all_shards)
expected = sorted(record['_final_idx'] for record in finalist if record['_effect_type'] in EFFECTS)
selected_bytes = sum(item['bytes'] for item in shard_index['shards'] if item['shard'] in shards)
estimate = 3 if PILOT else len(expected) * len(shards) / len(all_shards)
per_sample = 2.6e6 * (MAX_SIDE / 1280) ** 2  # đo: 1,12 + 1,12 + 0,36 MB ở 1280×720
download = estimate * 31e6 if PILOT else len(expected) / len(finalist) * selected_bytes  # ~31 MB một mẫu 4K
print(f'HALO {REVISION[:7]}: {len(finalist)} mẫu, {len(all_shards)} shard, {shard_index["total_bytes"] / 1e9:.1f} GB')
print(f'giữ {"/".join(EFFECTS)}: {"" if whole else "≈ "}{estimate:.0f} mẫu từ {len(shards)} shard | '
      f'tải ≈ {download / 1e9:.1f} GB thay vì {selected_bytes / 1e9:.0f} GB | '
      f'output ≈ {estimate * per_sample / 1e9:.1f} GB')
assert estimate * per_sample < 19e9, 'Output ước tính vượt 19 GB (Kaggle lưu tối đa 20 GB): giảm MAX_SIDE, EFFECTS hoặc N_SHARDS.'
if not BUILD.exists() and shutil.disk_usage('/tmp').free < estimate * per_sample * 1.2 + 2e9:
    # /tmp thiếu chỗ: dựng trong /kaggle/working, Cell 6 xoá từng scene sau khi tar của nó đọc lại khớp MD5
    BUILD = Path('/kaggle/working/_halo_build')
print(f'{WORKERS} process | dựng ở {BUILD} | /tmp trống {shutil.disk_usage("/tmp").free / 1e9:.0f} GB, '
      f'/kaggle/working trống {shutil.disk_usage("/kaggle/working").free / 1e9:.0f} GB')

## Cell 4 — Tải và thu nhỏ

Mỗi shard do một process đọc. Shard xong ghi `_shards/<shard>.json`; chạy lại cell trong cùng phiên thì shard đã xong được bỏ qua. Shard lỗi mạng được thử lại thêm một lượt.

In [ ]:
BUILD.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
reports, failed = build_all(shards, BUILD, HF_TOKEN, EFFECTS, MAX_SIDE, WORKERS, limit=3 if PILOT else None)
downloaded = sum(report['downloaded'] for report in reports.values())
seconds = time.perf_counter() - started
print(f'\n{len(reports)}/{len(shards)} shard | tải {downloaded / 1e9:.1f} GB trong {seconds / 60:.0f} phút '
      f'({downloaded / 1e6 / seconds:.0f} MB/s) | {sum(r["requests"] for r in reports.values()):,} request')
assert not failed, f'{len(failed)} shard vẫn lỗi sau hai lượt: chạy lại cell này (shard đã xong được bỏ qua).'

## Cell 5 — Kiểm tra

- mỗi shard phải đọc đủ số mẫu ghi trong `shard_index.json` (PILOT thì bỏ qua);
- `uid` và `final_idx` không được trùng. Chạy đủ 71 shard thì tập `final_idx` phải **đúng bằng** tập mẫu Reflective trong `finalist.json`;
- ba ảnh của một mẫu phải cùng cỡ, và cạnh dài phải bằng `MAX_SIDE`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

records = sorted((record for report in reports.values() for record in report['records']),
                 key=lambda record: record['final_idx'])
problems = [problem for report in reports.values() for problem in report['problems']]
if not PILOT:
    counts = {item['shard']: item['samples'] for item in shard_index['shards']}
    problems += [f'{name}: đọc được {report["samples"]} mẫu, shard_index.json ghi {counts[name]}'
                 for name, report in reports.items() if report['samples'] != counts[name]]
indices = [record['final_idx'] for record in records]
if len(set(indices)) != len(indices) or len({record['uid'] for record in records}) != len(records):
    problems.append('uid hoặc final_idx bị trùng')
if whole and not PILOT and indices != expected:
    problems.append(f'giữ {len(indices)} mẫu, finalist.json có {len(expected)} mẫu {EFFECTS}; '
                    f'thiếu {sorted(set(expected) - set(indices))[:10]}, thừa {sorted(set(indices) - set(expected))[:10]}')
alpha_kept, frame = 0, None
for record in records:
    stem = BUILD / record['scene'] / record['uid']
    meta = json.loads(Path(f'{stem}.json').read_text())
    for part in IMAGES:
        with Image.open(f'{stem}.{part}') as image:
            size = list(image.size)
        info = meta['images'][part]
        if size != info['size'] or max(size) != min(MAX_SIDE, max(info['source_size'])):
            problems.append(f'{stem}.{part}: {size}, json ghi {info}')
        alpha_kept += info['alpha']
    frame = frame or meta['images']['gt.png']['size']
    if len({tuple(meta['images'][part]['size']) for part in IMAGES}) != 1:
        problems.append(f'{stem}: ba ảnh khác cỡ')
for problem in problems[:20]:
    print('LỖI', problem)
assert records and not problems, f'{len(records)} mẫu, {len(problems)} lỗi: dừng trước khi đóng gói'
scenes = Counter(record['scene'] for record in records)
print(f'{len(records)} mẫu, {len(scenes)} scene, ảnh {frame[0]}×{frame[1]}, '
      f'{alpha_kept}/{3 * len(records)} ảnh có alpha thật (giữ RGBA), còn lại RGB')
print(', '.join(f'{scene} {count}' for scene, count in sorted(scenes.items())))

rng = np.random.default_rng(0)
chosen = [records[index] for index in sorted(rng.choice(len(records), min(4, len(records)), replace=False))]
figure, axes = plt.subplots(len(chosen), 3, figsize=(15, 2.9 * len(chosen)), squeeze=False)
for row, record in zip(axes, chosen):
    stem = BUILD / record['scene'] / record['uid']
    for axis, part in zip(row, IMAGES):
        image = np.asarray(Image.open(f'{stem}.{part}').convert('RGB'))
        if part == 'separate.png':
            image = np.clip(image.astype(np.float32) * 4, 0, 255).astype(np.uint8)  # lớp lóe tối, nhân 4 để thấy
        axis.imshow(image)
        axis.set_title(f'{record["uid"]} — {part}{" ×4" if part == "separate.png" else ""}', fontsize=8)
        axis.axis('off')
figure.tight_layout()
OUT.mkdir(parents=True, exist_ok=True)
figure.savefig(OUT / 'preview.jpg', dpi=70)
plt.show()

## Cell 6 — Đóng gói và mô tả

Mỗi scene thành một tar không nén (PNG đã nén sẵn). Mỗi file được băm MD5 lúc ghi, tar được **đọc lại toàn bộ** và so MD5 trước khi đổi `.tar.part` → `.tar`. Member chỉ mang tên file, nên khi Kaggle giải nén, dataset đọc thành `<scene>/<uid>.<phần>` mà không bị lặp tầng.

In [ ]:
width, height = frame
README = f'''# HALO — {"/".join(EFFECTS)} lens flare, {width}×{height}

Tập con của **HALO Synthetic Lens-Flare Dataset** (Zhang et al., đi kèm bài *UniSER: A Foundation Model
for Unified Soft Effects Removal*, CVPR 2026), lấy từ
https://huggingface.co/datasets/{REPO} (commit `{REVISION}`).

## Nội dung
- {len(records)} mẫu {"/".join(EFFECTS)} từ {len(scenes)} scene, render 3D bằng Blender, không có ảnh chụp thật.
- Mỗi mẫu gồm `<scene>/<uid>.gt.png` (không lóe), `.flare.png` (có lóe), `.separate.png` (chỉ lớp lóe)
  và `.json` (metadata gốc, thêm uid, shard, cỡ ảnh).
- `uid = sample_id + "_" + final_idx`. `sample_id` gốc không duy nhất: cùng scene, hiệu ứng và camera
  nhưng khác HDRI nền hoặc cấu hình đèn (xem cột `render_dir` của `halo_index.csv`).
- `halo_index.csv`: mỗi mẫu một dòng. `halo_inventory.tsv`: byte và MD5 của mọi file. `halo_build.json`:
  tham số dựng. `preview.jpg`: 4 mẫu (lớp separate nhân 4 để dễ thấy).

## Thay đổi so với bản gốc
- Chỉ giữ hiệu ứng {"/".join(EFFECTS)}.
- Thu nhỏ 3840×2160 → {width}×{height} bằng Lanczos (Pillow) và lưu PNG 8 bit.
- Bỏ kênh alpha khi nó bằng 255 ở mọi pixel ({3 * len(records) - alpha_kept}/{3 * len(records)} ảnh).
- Đổi tên file theo `uid`. Tên gốc nằm ở `source_names` trong json.

## Giấy phép
CC BY-NC-SA 4.0, chỉ cho nghiên cứu phi thương mại, theo điều khoản của HALO: trích dẫn UniSER khi công
bố kết quả, không phân phối lại dưới điều khoản thoáng hơn CC BY-NC-SA 4.0, và ghi nguồn HDRI nền
(Poly Haven, CC0; freepoly.org).

## Trích dẫn
```bibtex
@article{{zhang2025uniser,
  title={{UniSER: A Foundation Model for Unified Soft Effects Removal}},
  author={{Zhang, Jingdong and Zhang, Lingzhi and Liu, Qing and Chiu, Mang Tik and Barnes, Connelly and Wang, Yizhou and You, Haoran and Liu, Xiaoyang and Zhou, Yuqian and Lin, Zhe and others}},
  journal={{arXiv preprint arXiv:2511.14183}},
  year={{2025}}
}}
```
'''

inventory = pack_scenes(BUILD, OUT, remove_source=BUILD.is_relative_to('/kaggle/working'))
write_index(OUT / 'halo_index.csv', records, finalist)
with open(OUT / 'halo_inventory.tsv', 'w') as handle:
    handle.write('path\tbytes\tmd5\n')
    for path, (size, digest) in sorted(inventory.items()):
        handle.write(f'{path}\t{size}\t{digest}\n')
(OUT / 'halo_build.json').write_text(json.dumps({
    'repo': REPO, 'revision': REVISION, 'effects': EFFECTS, 'max_side': MAX_SIDE, 'frame': frame,
    'shards': len(shards), 'whole': whole, 'pilot': PILOT, 'samples': len(records), 'scenes': dict(sorted(scenes.items())),
    'alpha_kept': alpha_kept, 'files': len(inventory), 'bytes': sum(size for size, _ in inventory.values()),
    'downloaded_bytes': downloaded, 'build_seconds': round(seconds),
}, indent=2, ensure_ascii=False))
(OUT / 'README.md').write_text(README)
if BUILD.is_relative_to('/kaggle/working'):
    shutil.rmtree(BUILD)  # chỉ còn _shards/: mọi scene đã nằm trong tar đã kiểm
files = [path for path in Path('/kaggle/working').rglob('*') if path.is_file()]
print(f'Output sẽ lưu: {len(files)} file, {sum(path.stat().st_size for path in files) / 1e9:.2f} GB')
assert len(files) < 500, 'Output vẫn quá 500 file: New Dataset sẽ cắt mất phần còn lại.'

## Cell 7 — Tạo Kaggle dataset (cần secret `KAGGLE_API_TOKEN`)

Dataset chưa có thì tạo mới (riêng tư), có rồi thì thêm version. Không có secret thì cell chỉ in hướng dẫn tạo tay từ Output.

In [ ]:
KAGGLE_TOKEN, KAGGLE_KEY = secret('KAGGLE_API_TOKEN'), secret('KAGGLE_KEY')
if PILOT:
    print('PILOT: không tạo dataset. Đặt PILOT = False rồi Save Version → Save & Run All (Commit).')
elif not (KAGGLE_TOKEN or KAGGLE_KEY):
    print(f'Không có secret KAGGLE_API_TOKEN nên không tự tạo dataset. Khi Commit xong: tab Output → New Dataset, '
          f'tên {NAME}, để Private, License CC BY-NC-SA 4.0. Cửa sổ phải liệt kê {len(list(OUT.iterdir()))} file '
          f'trong {OUT.name}/; thấy hàng nghìn file là Output chưa được đóng gói.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'kaggle>=2.2'], check=True)
    env = dict(os.environ, **({'KAGGLE_API_TOKEN': KAGGLE_TOKEN} if KAGGLE_TOKEN else
                              {'KAGGLE_USERNAME': KAGGLE_OWNER, 'KAGGLE_KEY': KAGGLE_KEY}))
    command = shutil.which('kaggle') or 'kaggle'

    def kaggle(*args):
        result = subprocess.run([command, *args], env=env, capture_output=True, text=True)
        return result.returncode, (result.stdout + result.stderr).strip()

    (OUT / 'dataset-metadata.json').write_text(json.dumps({
        'id': DATASET, 'title': TITLE, 'licenses': [{'name': 'CC-BY-NC-SA-4.0'}],
        'subtitle': f'HALO lens-flare triplets ({"/".join(EFFECTS)}), {width}x{height}'[:80],
        'description': README,
    }, indent=2, ensure_ascii=False))  # CLI không upload file này
    exists = kaggle('datasets', 'status', DATASET)[0] == 0
    if exists:
        code, text = kaggle('datasets', 'version', '-p', str(OUT), '-m', f'HALO {REVISION[:7]}: {len(records)} mẫu')
    else:
        code, text = kaggle('datasets', 'create', '-p', str(OUT))  # mặc định riêng tư
    print(text[-3000:])
    # CLI in "... error: ..." mà vẫn trả mã 0
    assert code == 0 and 'error' not in text.lower(), 'Tạo dataset thất bại (xem dòng trên)'
    status = ''
    for _ in range(90):  # Kaggle giải nén các tar sau khi nhận xong
        status = (kaggle('datasets', 'status', DATASET)[1].splitlines() or [''])[-1].strip().lower()
        if status in ('ready', 'error'):
            break
        time.sleep(20)
    print(f'trạng thái: {status} | https://www.kaggle.com/datasets/{DATASET}')

## Sau khi dựng xong

**Kiểm dataset.** Mở trang dataset: mỗi `<scene>.tar` phải thành thư mục `<scene>/`, tổng cộng
4 × (số mẫu) file như `halo_inventory.tsv`. Nếu dataset tạo tay từ Output thì mọi thứ nằm thêm một tầng
`halo_reflective_1280/`.

**Đọc trong notebook train.** Gắn dataset vào notebook train, rồi:

```python
import csv
from pathlib import Path

# halo_index.csv nằm ở gốc dataset (hoặc trong halo_reflective_1280/ nếu tạo từ Output)
ROOT = next(path.parent for depth in range(1, 6)
            for path in Path('/kaggle/input').glob('/'.join(['*'] * depth) + '/halo_index.csv'))
rows = list(csv.DictReader(open(ROOT / 'halo_index.csv')))
row = rows[0]
gt, flare, separate = (ROOT / row['scene'] / f"{row['uid']}.{part}.png" for part in ('gt', 'flare', 'separate'))
```

Nên chia train/valid theo `scene` (hoặc theo HDRI trong `render_dir`), không chia theo mẫu: các mẫu cùng
scene có cùng hình học, chỉ khác camera, đèn hoặc nền.